# ✈️ Flight Fare Prediction
**Model:** XGBoost with hand-engineered features  
**Dataset:** EaseMyTrip domestic Indian flights (~300k rows)  
**Target:** Flight price (INR)  

**Feature engineering highlights:**
- Booking lead time (days before departure)
- Departure/arrival hour bins
- Duration in minutes
- Weekend flag
- Airline target encoding

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q xgboost scikit-learn pandas matplotlib seaborn shap

In [ ]:
# Download dataset (Kaggle EaseMyTrip or use sample)
# Option A: Kaggle API
# !pip install kaggle
# !kaggle datasets download -d shubhambathwal/flight-price-prediction

# Option B: Use a sample CSV already in the repo
import pandas as pd
import numpy as np

# Generate a synthetic sample for demo if no data file present
np.random.seed(42)
n = 5000
airlines = ['IndiGo', 'Air India', 'Jet Airways', 'SpiceJet', 'Vistara']
cities   = ['Delhi', 'Mumbai', 'Bangalore', 'Kolkata', 'Chennai', 'Hyderabad']

df = pd.DataFrame({
    'airline':     np.random.choice(airlines, n),
    'source':      np.random.choice(cities, n),
    'destination': np.random.choice(cities, n),
    'stops':       np.random.choice(['zero','one','two_or_more'], n, p=[0.5,0.35,0.15]),
    'duration':    [f"{np.random.randint(1,8)}h {np.random.randint(0,59)}m" for _ in range(n)],
    'departure_time': [f"{np.random.randint(0,23):02d}:{np.random.randint(0,59):02d}" for _ in range(n)],
    'arrival_time':   [f"{np.random.randint(0,23):02d}:{np.random.randint(0,59):02d}" for _ in range(n)],
    'date_of_journey': pd.date_range('2024-01-01', periods=n, freq='2H').strftime('%d/%m/%Y'),
})
df['price'] = (
    3000
    + df['stops'].map({'zero': 0, 'one': 2000, 'two_or_more': 4000})
    + np.random.normal(0, 1500, n)
).clip(1500, 50000).astype(int)

import os
os.makedirs('data', exist_ok=True)
df.to_csv('data/flight_fare.csv', index=False)
print(f'Dataset: {df.shape} | Price range: ₹{df.price.min():,} – ₹{df.price.max():,}')
df.head()

In [ ]:
# EDA
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df['price'], bins=50, ax=axes[0]); axes[0].set_title('Price Distribution')
sns.boxplot(data=df, x='stops', y='price', ax=axes[1]); axes[1].set_title('Price by Stops')
sns.boxplot(data=df, x='airline', y='price', ax=axes[2]); axes[2].set_title('Price by Airline')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Feature engineering + training
import sys; sys.path.insert(0, '.')
from modules.regression.flight_fare.model import engineer_features, FEATURE_COLS, train
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

metrics = train('data/flight_fare.csv')
print('Training complete:', metrics)

In [ ]:
# SHAP feature importance
import shap, pickle
import pandas as pd

with open('weights/flight_fare.pkl', 'rb') as f:
    model = pickle.load(f)

df_feat = engineer_features(pd.read_csv('data/flight_fare.csv'))
X = df_feat[FEATURE_COLS].dropna()

explainer  = shap.TreeExplainer(model)
shap_vals  = explainer.shap_values(X[:200])
shap.summary_plot(shap_vals, X[:200], plot_type='bar', show=True)

In [ ]:
# Single prediction
from modules.regression.flight_fare.model import predict

sample = {
    'airline_encoded': 0, 'source_encoded': 0, 'destination_encoded': 1,
    'stops': 1, 'duration_minutes': 180, 'departure_hour': 9,
    'arrival_hour': 12, 'days_before_departure': 14,
    'is_weekend': 0, 'is_morning_flight': 1
}
price, ms = predict(sample)
print(f'Predicted fare: ₹{price:,.0f} ({ms:.1f}ms)')